<a href="https://colab.research.google.com/github/randyliu078-bit/qss20-final-project/blob/main/code/02_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# 02_clean.ipynb — QSS 20 Final Project — Randy Liu
# Cleans and prepares ACS, PSEO, and QCEW data
# ============================================================


# ============================================================
# Helper functions — defined at top per coding standards
# ============================================================

def pull_acs(api_key):
    """Pull ACS degree attainment data for CT, MA, NY. Returns cleaned DataFrame."""
    url = (
        "https://api.census.gov/data/2022/acs/acs1/subject"
        "?get=NAME,S1501_C02_006E,S1501_C02_007E,S1501_C02_008E,"
        "S1501_C02_009E,S1501_C02_010E,S1501_C02_011E"
        f"&for=state:09,25,36&key={api_key}"
    )
    df = pd.DataFrame(requests.get(url).json()[1:],
                      columns=["state_name","age_25_29","age_30_34","age_35_44",
                                "age_45_54","age_55_64","age_65plus","state_fips"])
    df = df.replace("-888888888", None)
    for col in ["age_25_29","age_30_34","age_35_44","age_45_54","age_55_64","age_65plus"]:
        df[col] = pd.to_numeric(df[col])
    return df

def pull_pseo(state="ct"):
    """Pull PSEO earnings file for a given state. Returns bachelor's-only DataFrame."""
    url = f"https://lehd.ces.census.gov/data/pseo/R2024Q4/{state}/pseoe_{state}.csv.gz"
    df = pd.read_csv(io.BytesIO(requests.get(url).content),
                     compression="gzip", low_memory=False)
    return df[df["degree_level"] == 5].copy()

def pull_qcew(area_fips="09000"):
    """Pull QCEW annual data for a given area FIPS. Returns cleaned DataFrame."""
    qcew_cols = [
        'area_fips','own_code','industry_code','agglvl_code','size_code',
        'year','qtr','disclosure_code','annual_avg_estabs','annual_avg_emplvl',
        'total_annual_wages','taxable_annual_wages','annual_contributions',
        'annual_avg_wkly_wage','avg_annual_pay','lq_disclosure_code',
        'lq_annual_avg_estabs','lq_annual_avg_emplvl','lq_total_annual_wages',
        'lq_taxable_annual_wages','lq_annual_contributions','lq_annual_avg_wkly_wage',
        'lq_avg_annual_pay','oty_disclosure_code','oty_annual_avg_estabs_chg',
        'oty_annual_avg_estabs_pct_chg','oty_annual_avg_emplvl_chg',
        'oty_annual_avg_emplvl_pct_chg','oty_total_annual_wages_chg',
        'oty_total_annual_wages_pct_chg','oty_taxable_annual_wages_chg',
        'oty_taxable_annual_wages_pct_chg','oty_annual_contributions_chg',
        'oty_annual_contributions_pct_chg','oty_annual_avg_wkly_wage_chg',
        'oty_annual_avg_wkly_wage_pct_chg','oty_avg_annual_pay_chg',
        'oty_avg_annual_pay_pct_chg'
    ]
    return pd.read_csv(f"https://data.bls.gov/cew/data/api/2022/a/area/{area_fips}.csv",
                       header=None, names=qcew_cols, skiprows=1)

In [5]:


import pandas as pd
import requests
import io

API_KEY = "84be3d1f35c778150bb57bc7f815f88136459eac"

# ---- Re-pull ACS data ----
acs_url = "https://api.census.gov/data/2022/acs/acs1/subject?get=NAME,S1501_C02_006E,S1501_C02_007E,S1501_C02_008E,S1501_C02_009E,S1501_C02_010E,S1501_C02_011E&for=state:09,25,36&key=" + API_KEY
acs_df = pd.DataFrame(requests.get(acs_url).json()[1:],
                      columns=requests.get(acs_url).json()[0])

# Rename columns to meaningful names
acs_df.columns = ["state_name","age_25_29","age_30_34","age_35_44","age_45_54","age_55_64","age_65plus","state_fips"]
acs_df = acs_df.replace("-888888888", None)

# Convert to numeric
for col in ["age_25_29","age_30_34","age_35_44","age_45_54","age_55_64","age_65plus"]:
    acs_df[col] = pd.to_numeric(acs_df[col])

print("ACS cleaned:")
print(acs_df)

# ---- Re-pull QCEW data ----
qcew_df = pd.read_csv("https://data.bls.gov/cew/data/api/2022/a/area/09000.csv")
qcew_df = qcew_df[qcew_df["industry_code"] == 1013]  # NAICS 3364 aerospace
print("\nQCEW cleaned (aerospace rows):")
print(qcew_df[["area_fips","industry_code","annual_avg_emplvl","annual_avg_wkly_wage"]].head())

# ---- Re-pull PSEO data ----
pseo_df = pd.read_csv(io.BytesIO(requests.get(
    "https://lehd.ces.census.gov/data/pseo/R2024Q4/ct/pseoe_ct.csv.gz"
).content), compression="gzip", low_memory=False)

# Filter to bachelor's degrees only (degree_level == 5)
pseo_clean = pseo_df[pseo_df["degree_level"] == 5].copy()
print(f"\nPSEO cleaned: {pseo_clean.shape[0]} rows (bachelor's only)")
print(pseo_clean[["institution","cipcode","y1_earnings","y5_earnings"]].head())

ACS cleaned:
      state_name  age_25_29  age_30_34  age_35_44  age_45_54  age_55_64  \
0    Connecticut        NaN        3.9        4.5       25.9       15.9   
1  Massachusetts        NaN        4.4        4.3       23.0       14.3   
2       New York        NaN        6.0        6.1       24.6       14.5   

   age_65plus state_fips  
0         7.7         09  
1         7.4         25  
2         8.8         36  

QCEW cleaned (aerospace rows):
Empty DataFrame
Columns: [area_fips, industry_code, annual_avg_emplvl, annual_avg_wkly_wage]
Index: []

PSEO cleaned: 7178 rows (bachelor's only)


KeyError: "['y1_earnings', 'y5_earnings'] not in index"

In [ ]:
# Check actual column names in PSEO and QCEW
print("PSEO columns:", pseo_clean.columns.tolist())
print("\nQCEW industry codes sample:", qcew_df["industry_code"].unique()[:20])


In [ ]:
# ---- Fixed PSEO display with correct column names ----
print("PSEO cleaned: 7178 bachelor's degree rows")
print(pseo_clean[["institution","cipcode","y1_p50_earnings","y5_p50_earnings","y10_p50_earnings"]].head(10))

# ---- Fixed QCEW filter ----
print("\nAll QCEW industry codes:", qcew_df["industry_code"].unique()[:30])
# Show total CT employment and wages instead
qcew_summary = qcew_df[qcew_df["agglvl_code"] == 50][["area_fips","annual_avg_emplvl","annual_avg_wkly_wage","total_annual_wages"]]
print("\nQCEW CT summary:")
print(qcew_summary.head())

# ---- Save cleaned data ----
acs_df.to_csv("acs_clean.csv", index=False)
pseo_clean.to_csv("pseo_clean.csv", index=False)
qcew_df.to_csv("qcew_clean.csv", index=False)
print("\nAll cleaned files saved!")

In [ ]:
print(qcew_df.columns.tolist())
print(qcew_df.head(2))

In [ ]:
# ---- Fixed QCEW pull and clean ----
qcew_df = pd.read_csv("https://data.bls.gov/cew/data/api/2022/a/area/09000.csv",
                       skiprows=1)
print("QCEW shape:", qcew_df.shape)
print("agglvl_code values:", qcew_df["agglvl_code"].unique())
print(qcew_df.head(3))

In [ ]:
print(qcew_df.columns.tolist()[:10])
print(qcew_df.head(2))

In [ ]:
# ---- Fixed QCEW with correct column names ----
qcew_cols = ['area_fips','own_code','industry_code','agglvl_code','size_code',
             'year','qtr','disclosure_code','annual_avg_estabs','annual_avg_emplvl',
             'total_annual_wages','taxable_annual_wages','annual_contributions',
             'annual_avg_wkly_wage','avg_annual_pay','lq_disclosure_code',
             'lq_annual_avg_estabs','lq_annual_avg_emplvl','lq_total_annual_wages',
             'lq_taxable_annual_wages','lq_annual_contributions','lq_annual_avg_wkly_wage',
             'lq_avg_annual_pay','oty_disclosure_code','oty_annual_avg_estabs_chg',
             'oty_annual_avg_estabs_pct_chg','oty_annual_avg_emplvl_chg',
             'oty_annual_avg_emplvl_pct_chg','oty_total_annual_wages_chg',
             'oty_total_annual_wages_pct_chg','oty_taxable_annual_wages_chg',
             'oty_taxable_annual_wages_pct_chg','oty_annual_contributions_chg',
             'oty_annual_contributions_pct_chg','oty_annual_avg_wkly_wage_chg',
             'oty_annual_avg_wkly_wage_pct_chg','oty_avg_annual_pay_chg',
             'oty_avg_annual_pay_pct_chg']

qcew_df = pd.read_csv("https://data.bls.gov/cew/data/api/2022/a/area/09000.csv",
                      header=None, names=qcew_cols, skiprows=1)

print("Shape:", qcew_df.shape)
print("agglvl_code values:", qcew_df["agglvl_code"].unique())
qcew_summary = qcew_df[qcew_df["agglvl_code"] == 50][["area_fips","annual_avg_emplvl","annual_avg_wkly_wage"]]
print(qcew_summary.head())
qcew_df.to_csv("qcew_clean.csv", index=False)
print("QCEW saved!")